# Phase 8 — Robustness studies (fourth review round)

Same protocol and shared code as Phases 1–7. Set `PART` in cell 1 to run one group:

* **whu_random / whu_disjoint:** all eight backbone variants on WHU-Hi HanChuan, random 5% and patch-disjoint splits (10 seeds; test sets capped at 50,000 pixels).
* **misc:** a deeper, wider host (4 residual blocks of 128 channels) on IP random and KSC patch-disjoint (10 seeds); learning rates 3e-4 and 3e-3 on IP random (5 seeds); PCA fitted on training pixels only on KSC patch-disjoint (15 seeds); scarcity imposed on KSC's three hardest classes (15 seeds).
* **hard_pu:** scarcity imposed on Pavia University's three hardest classes (15 seeds).
* **blocks_pu / blocks_ksc:** block-disjoint splits of PU and KSC (training pixels scattered over many 16x16 blocks; 15 seeds); `blocks_ksc` also reruns GCT on IP random seeds 42-46 to log its learned gate.
* **lr_high:** learning rate 3e-3 on IP random, seeds 47-56, extending the five `misc` seeds to 15.

Output: `/kaggle/working/phase8/runs.csv`, `summary.csv`.

In [ ]:
# =============================================================================
# Cell 1: Setup & configuration
# =============================================================================
import os, sys, time, json, glob, copy, random, subprocess, gc

try:
    import thop
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop'], check=False)
    try:
        import thop
    except ImportError:
        thop = None

import numpy as np
import scipy.io as sio
from scipy import signal
from scipy.ndimage import zoom
import pywt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

SMOKE = os.environ.get('PHASE8_SMOKE') == '1'   # local synthetic-data test only

# ── Protocol (do not change between models) ──
SEEDS          = list(range(42, 57))
EPOCHS         = 100
BATCH_SIZE     = 64
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
STEP_SIZE      = 40
STEP_GAMMA     = 0.5
WINDOW         = 15
TRAIN_RATIO    = 0.05
PCA_COMPONENTS = 30
MAX_TEST       = 50000

# ── What to run ──
BLOCKS = ['plain', 'se_gamma', 'eca_gamma', 'cbam', 'gct', 'srm', 'snsm_gamma', 'snsm']
S10, S15 = list(range(42, 52)), list(range(42, 57))
# (name, dataset, split, models, seeds, learning rate, features)
ALL_STUDIES = {
    'whu_random':   [('whu_random', 'WHU_Hi_HanChuan', 'random', BLOCKS, S10, 1e-3, 'scene')],
    'whu_disjoint': [('whu_disjoint', 'WHU_Hi_HanChuan', 'disjoint', BLOCKS, S10, 1e-3, 'scene')],
    'misc': [('deep_ip_random', 'IndianPines', 'random', ['deep_' + m for m in BLOCKS], S10, 1e-3, 'scene'),
             ('deep_ksc_disjoint', 'KSC', 'disjoint', ['deep_' + m for m in BLOCKS], S10, 1e-3, 'scene'),
             ('lr3e-4_ip_random', 'IndianPines', 'random', BLOCKS, S10[:5], 3e-4, 'scene'),
             ('lr3e-3_ip_random', 'IndianPines', 'random', BLOCKS, S10[:5], 3e-3, 'scene'),
             ('trainpca_ksc_disjoint', 'KSC', 'disjoint', BLOCKS, S15, 1e-3, 'train'),
             ('hard_ksc', 'KSC', 'hard', BLOCKS, S15, 1e-3, 'scene')],
    'hard_pu': [('hard_pu', 'PaviaU', 'hard', BLOCKS, S15, 1e-3, 'scene')],
    'blocks_pu': [('blocks_pu', 'PaviaU', 'blocks', BLOCKS, S15, 1e-3, 'scene')],
    'blocks_ksc': [('blocks_ksc', 'KSC', 'blocks', BLOCKS, S15, 1e-3, 'scene'),
                   ('gct_gate_ip', 'IndianPines', 'random', ['gct'], S10[:5], 1e-3, 'scene')],
    'lr_high': [('lr3e-3_ip_random', 'IndianPines', 'random', BLOCKS, S15[5:], 3e-3, 'scene')],   # seeds 47-56 (fifth review round)
}
PART = 'misc'
STUDIES = ALL_STUDIES[PART]
# Hardest classes (lowest plain-CNN accuracy on random 5% splits), 1-based labels -> training pixels 1, 1, 2
HARD = {'PaviaU': {9: 1, 4: 1, 8: 2},     # Shadows, Trees, Self-Blocking Bricks
        'KSC': {4: 1, 8: 1, 5: 2}}        # Cabbage palm/oak hammock, Graminoid marsh, Slash pine

OUT_DIR = '/kaggle/working/phase8'
if SMOKE:
    SEEDS, EPOCHS, MAX_TEST = [42, 43], 2, 300
    OUT_DIR = os.environ['PHASE8_OUT']
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
env = {'torch': torch.__version__, 'device': str(device),
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       'thop': thop is not None, 'smoke': SMOKE}
with open(os.path.join(OUT_DIR, 'env.json'), 'w') as f:
    json.dump(env, f, indent=2)
print(env)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# =============================================================================
# Cell 2: Data loading, preprocessing and splits
# =============================================================================
from scipy.ndimage import maximum_filter

DATASET_INFO = {
    'IndianPines':     {'data_key': 'indian_pines_corrected', 'gt_key': 'indian_pines_gt'},
    'PaviaU':          {'data_key': 'paviaU',                 'gt_key': 'paviaU_gt'},
    'Salinas':         {'data_key': 'salinas_corrected',      'gt_key': 'salinas_gt'},
    'Botswana':        {'data_key': 'Botswana',               'gt_key': 'Botswana_gt'},
    'KSC':             {'data_key': 'KSC',                    'gt_key': 'KSC_gt'},
    'WHU_Hi_HanChuan': {'data_key': 'WHU_Hi_HanChuan',        'gt_key': 'WHU_Hi_HanChuan_gt'},
}
CLASS_NAMES = {
    'IndianPines': ['Alfalfa', 'Corn-notill', 'Corn-mintill', 'Corn', 'Grass-pasture',
                    'Grass-trees', 'Grass-pasture-mowed', 'Hay-windrowed', 'Oats',
                    'Soybean-notill', 'Soybean-mintill', 'Soybean-clean', 'Wheat', 'Woods',
                    'Bldg-Grass-Trees-Drives', 'Stone-Steel-Towers'],
    'PaviaU': ['Asphalt', 'Meadows', 'Gravel', 'Trees', 'Painted metal sheets', 'Bare Soil',
               'Bitumen', 'Self-Blocking Bricks', 'Shadows'],
    'KSC': ['Scrub', 'Willow swamp', 'Cabbage palm hammock', 'Cabbage palm/oak hammock',
            'Slash pine', 'Oak/broadleaf hammock', 'Hardwood swamp', 'Graminoid marsh',
            'Spartina marsh', 'Cattail marsh', 'Salt marsh', 'Mud flats', 'Water'],
    'Botswana': ['Water', 'Hippo grass', 'Floodplain grasses 1', 'Floodplain grasses 2',
                 'Reeds', 'Riparian', 'Firescar', 'Island interior', 'Acacia woodlands',
                 'Acacia shrublands', 'Acacia grasslands', 'Short mopane', 'Mixed mopane',
                 'Exposed soils'],
}
DATA_DIRS = ['/kaggle/input/datasets/devgurucodes/datasets-hsi-all',
             '/kaggle/input/datasets-hsi-all']


def _find(fname):
    for d in DATA_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError(f'{fname} not found under /kaggle/input — add the datasets_hsi_all input')


def _synthetic(name):
    # Small fake scene with class-specific spectra (local smoke test only).
    rng = np.random.RandomState(sum(map(ord, name)))
    h, w, b, k = 40, 36, 48, 5
    gt = np.zeros((h, w), dtype=np.int32)
    for c in range(1, k + 1):
        r0, c0 = rng.randint(0, h - 10), rng.randint(0, w - 10)
        gt[r0:r0 + 10, c0:c0 + 10] = c
    sig = rng.rand(k + 1, b) * 100
    img = sig[gt] + rng.randn(h, w, b) * 5
    return img.astype(np.float32), gt


def load_dataset(name):
    if SMOKE:
        return _synthetic(name)
    info = DATASET_INFO[name]
    img = sio.loadmat(_find(f'{name}_data.mat'))[info['data_key']]
    gt = sio.loadmat(_find(f'{name}_gt.mat'))[info['gt_key']]
    # KSC is stored as uint16 but holds small negative values (-27..-1 wrap to 65509..65535): read it as int16.
    if name == 'KSC':
        img = img.view(np.int16)
    return img, gt


def wiener_denoise(X):
    out = np.zeros(X.shape, dtype=np.float32)
    for b in range(X.shape[2]):
        out[:, :, b] = signal.wiener(X[:, :, b].astype(np.float64), mysize=(5, 5)).astype(np.float32)
    return out


def apply_pca(X, n):
    h, w, b = X.shape
    return PCA(n_components=n).fit_transform(X.reshape(-1, b)).reshape(h, w, n).astype(np.float32)


def wavelet_features(X):
    h, w, b = X.shape
    ll = np.zeros((h, w, b), dtype=np.float32)
    for i in range(b):
        LL, _ = pywt.dwt2(X[:, :, i].astype(np.float64), 'db4')
        ll[:, :, i] = zoom(LL, (h / LL.shape[0], w / LL.shape[1]), order=1)[:h, :w]
    return np.concatenate([X, ll], axis=2)


def preprocess(img):
    return wavelet_features(apply_pca(wiener_denoise(img), PCA_COMPONENTS))


def split_centers(gt, seed, ratio=TRAIN_RATIO):
    # Same per-class shuffling as the SNSM notebook, so seeds 42-46 give identical splits.
    rng = np.random.RandomState(seed)
    tr_xy, tr_y, te_xy, te_y = [], [], [], []
    for c in np.unique(gt[gt > 0]):
        idx = np.argwhere(gt == c)
        rng.shuffle(idx)
        n = max(1, int(len(idx) * ratio))
        tr_xy.append(idx[:n]); tr_y.append(np.full(n, c - 1))
        te_xy.append(idx[n:]); te_y.append(np.full(len(idx) - n, c - 1))
    tr_xy, tr_y = np.concatenate(tr_xy), np.concatenate(tr_y)
    te_xy, te_y = np.concatenate(te_xy), np.concatenate(te_y)
    if len(te_y) > MAX_TEST:
        keep = np.sort(np.random.RandomState(seed).choice(len(te_y), MAX_TEST, replace=False))
        te_xy, te_y = te_xy[keep], te_y[keep]
    return tr_xy, tr_y.astype(np.int64), te_xy, te_y.astype(np.int64)


def split_disjoint(gt, seed, ratio=TRAIN_RATIO):
    # Spatially disjoint split: each class's training pixels are the compact cluster of its pixels nearest
    # a random centre; test pixels within WINDOW-1 (Chebyshev) of any training pixel are dropped, so no test
    # patch overlaps a training patch.
    rng = np.random.RandomState(seed)
    tr_xy, tr_y = [], []
    for c in np.unique(gt[gt > 0]):
        idx = np.argwhere(gt == c)
        n = max(1, int(len(idx) * ratio))
        centre = idx[rng.randint(len(idx))]
        order = np.argsort(((idx - centre) ** 2).sum(1), kind='stable')
        tr_xy.append(idx[order[:n]]); tr_y.append(np.full(n, c - 1))
    tr_xy, tr_y = np.concatenate(tr_xy), np.concatenate(tr_y)
    train_mask = np.zeros(gt.shape, dtype=bool)
    train_mask[tr_xy[:, 0], tr_xy[:, 1]] = True
    near = maximum_filter(train_mask, size=2 * (WINDOW - 1) + 1)
    te_xy = np.argwhere((gt > 0) & ~near)
    te_y = gt[te_xy[:, 0], te_xy[:, 1]] - 1
    return tr_xy, tr_y.astype(np.int64), te_xy, te_y.astype(np.int64)


def split_counts(gt, seed, n_small=15, n_other=50, k_small=3):
    # Random split with the standard IP split's per-class training counts: n_small pixels for the k_small
    # smallest classes, n_other for every other class; the remaining labelled pixels are the test set.
    rng = np.random.RandomState(seed)
    classes = np.unique(gt[gt > 0])
    sizes = np.array([(gt == c).sum() for c in classes])
    small = set(classes[np.argsort(sizes, kind='stable')[:k_small]])
    tr_xy, tr_y, te_xy, te_y = [], [], [], []
    for c in classes:
        idx = np.argwhere(gt == c)
        rng.shuffle(idx)
        n = n_small if c in small else n_other
        assert n < len(idx), (c, n, len(idx))
        tr_xy.append(idx[:n]); tr_y.append(np.full(n, c - 1))
        te_xy.append(idx[n:]); te_y.append(np.full(len(idx) - n, c - 1))
    return (np.concatenate(tr_xy), np.concatenate(tr_y).astype(np.int64),
            np.concatenate(te_xy), np.concatenate(te_y).astype(np.int64))


def split_scarcity(gt, seed, small_counts, ratio=TRAIN_RATIO):
    # Random split: the k smallest classes get small_counts[i] training pixels (i-th smallest), every other
    # class max(1, int(ratio * size)); the remaining labelled pixels are the test set.
    rng = np.random.RandomState(seed)
    classes = np.unique(gt[gt > 0])
    sizes = np.array([(gt == c).sum() for c in classes])
    fixed = {classes[i]: n for i, n in zip(np.argsort(sizes, kind='stable'), small_counts)}
    tr_xy, tr_y, te_xy, te_y = [], [], [], []
    for c in classes:
        idx = np.argwhere(gt == c)
        rng.shuffle(idx)
        n = fixed.get(c, max(1, int(len(idx) * ratio)))
        assert 0 < n < len(idx), (c, n, len(idx))
        tr_xy.append(idx[:n]); tr_y.append(np.full(n, c - 1))
        te_xy.append(idx[n:]); te_y.append(np.full(len(idx) - n, c - 1))
    return (np.concatenate(tr_xy), np.concatenate(tr_y).astype(np.int64),
            np.concatenate(te_xy), np.concatenate(te_y).astype(np.int64))


def split_hard(gt, seed, fixed, ratio=TRAIN_RATIO):
    # Random split in which the listed classes (1-based label -> training pixels) get fixed counts.
    rng = np.random.RandomState(seed)
    tr_xy, tr_y, te_xy, te_y = [], [], [], []
    for c in np.unique(gt[gt > 0]):
        idx = np.argwhere(gt == c)
        rng.shuffle(idx)
        n = fixed.get(int(c), max(1, int(len(idx) * ratio)))
        assert 0 < n < len(idx), (c, n, len(idx))
        tr_xy.append(idx[:n]); tr_y.append(np.full(n, c - 1))
        te_xy.append(idx[n:]); te_y.append(np.full(len(idx) - n, c - 1))
    return (np.concatenate(tr_xy), np.concatenate(tr_y).astype(np.int64),
            np.concatenate(te_xy), np.concatenate(te_y).astype(np.int64))


def split_blocks(gt, seed, ratio=TRAIN_RATIO, tile=16):
    # Block-disjoint split: the scene is cut into tile x tile blocks visited in one random order per seed;
    # each class takes its pixels block by block until it has 5% of them, so training pixels are scattered
    # over many small blocks instead of one cluster. Test pixels within WINDOW-1 (Chebyshev) of any training
    # pixel are dropped, as in split_disjoint, so no test patch overlaps a training patch.
    rng = np.random.RandomState(seed)
    n_rows, n_cols = -(-gt.shape[0] // tile), -(-gt.shape[1] // tile)
    rank = np.empty(n_rows * n_cols, dtype=np.int64)
    rank[rng.permutation(len(rank))] = np.arange(len(rank))
    tr_xy, tr_y = [], []
    for c in np.unique(gt[gt > 0]):
        idx = np.argwhere(gt == c)
        n = max(1, int(len(idx) * ratio))
        order = np.argsort(rank[(idx[:, 0] // tile) * n_cols + idx[:, 1] // tile], kind='stable')
        tr_xy.append(idx[order[:n]]); tr_y.append(np.full(n, c - 1))
    tr_xy, tr_y = np.concatenate(tr_xy), np.concatenate(tr_y)
    train_mask = np.zeros(gt.shape, dtype=bool)
    train_mask[tr_xy[:, 0], tr_xy[:, 1]] = True
    near = maximum_filter(train_mask, size=2 * (WINDOW - 1) + 1)
    te_xy = np.argwhere((gt > 0) & ~near)
    te_y = gt[te_xy[:, 0], te_xy[:, 1]] - 1
    return tr_xy, tr_y.astype(np.int64), te_xy, te_y.astype(np.int64)


def cap_test(split, seed):
    # Same 50,000-pixel test cap as the random splits (only WHU-Hi exceeds it here).
    tr_xy, ytr, te_xy, yte = split
    if len(yte) > MAX_TEST:
        keep = np.sort(np.random.RandomState(seed).choice(len(yte), MAX_TEST, replace=False))
        te_xy, yte = te_xy[keep], yte[keep]
    return tr_xy, ytr, te_xy, yte


def features_train_pca(filtered, tr_xy):
    # PCA fitted on the training pixels only (inductive), then the same wavelet step as preprocess().
    h, w, b = filtered.shape
    pca = PCA(n_components=PCA_COMPONENTS).fit(filtered[tr_xy[:, 0], tr_xy[:, 1]])
    return wavelet_features(pca.transform(filtered.reshape(-1, b)).reshape(h, w, PCA_COMPONENTS).astype(np.float32))


def pad(X):
    m = WINDOW // 2
    return np.pad(X, ((m, m), (m, m), (0, 0))).astype(np.float32)


def extract_patches(padded, coords):
    out = np.empty((len(coords), padded.shape[2], WINDOW, WINDOW), dtype=np.float32)
    for i, (r, c) in enumerate(coords):
        out[i] = padded[r:r + WINDOW, c:c + WINDOW, :].transpose(2, 0, 1)
    return out

print('Data utilities ready.')

In [ ]:
# =============================================================================
# Cell 3: Models
# =============================================================================
# All modulation variants share one backbone (identical to SNSM_CNN):
#   conv3x3(bands->64)+BN+ReLU -> 2 residual blocks -> GAP -> dropout 0.4 -> FC
# Each residual block: conv-BN-ReLU -> [modulation] -> conv-BN -> +x -> ReLU
# Variants differ ONLY in the modulation step:
#   plain          : none
#   se_gamma       : SE-style, GAP -> FC -> sigmoid scale (gamma only)
#   eca_gamma      : ECA, GAP -> Conv1d(k=3) across channels -> sigmoid scale (gamma only)
#   cbam           : CBAM, channel gate (shared MLP on avg+max pool) then 7x7 spatial gate
#   fc_gamma_beta  : GAP -> FC -> sigmoid scale + linear shift (previous SSMRB)
#   snsm_gamma     : spectral-neighbourhood descriptors -> Conv1d -> scale only
#   snsm           : spectral-neighbourhood descriptors -> Conv1d -> scale + shift (proposed)

class SEGamma(nn.Module):
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(ch, reduction)
        self.fc_gamma = nn.Linear(reduction, ch)

    def forward(self, x):
        z = F.relu(self.fc1(x.mean(dim=(2, 3))))
        return x * torch.sigmoid(self.fc_gamma(z))[:, :, None, None]


class ECAGamma(nn.Module):
    # Wang et al. (2020); adaptive kernel size for C=64 is 3.
    def __init__(self, ch, k=3):
        super().__init__()
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)

    def forward(self, x):
        z = self.conv(x.mean(dim=(2, 3)).unsqueeze(1)).squeeze(1)
        return x * torch.sigmoid(z)[:, :, None, None]


class CBAM(nn.Module):
    # Woo et al. (2018): channel attention (reduction 16) followed by 7x7 spatial attention.
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(ch, ch // reduction), nn.ReLU(), nn.Linear(ch // reduction, ch))
        self.spatial = nn.Conv2d(2, 1, 7, padding=3, bias=False)

    def forward(self, x):
        gate = torch.sigmoid(self.mlp(x.mean(dim=(2, 3))) + self.mlp(x.amax(dim=(2, 3))))
        x = x * gate[:, :, None, None]
        s = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return x * torch.sigmoid(self.spatial(s))


class GCT(nn.Module):
    # Yang et al. (2020), l2 mode: embedding -> channel normalisation -> 1 + tanh gate; identity at init.
    def __init__(self, ch, eps=1e-5):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1, ch, 1, 1))
        self.gamma = nn.Parameter(torch.zeros(1, ch, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, ch, 1, 1))
        self.eps = eps

    def forward(self, x):
        emb = (x.pow(2).sum(dim=(2, 3), keepdim=True) + self.eps).sqrt() * self.alpha
        norm = self.gamma / (emb.pow(2).mean(dim=1, keepdim=True) + self.eps).sqrt()
        return x * (1.0 + torch.tanh(emb * norm + self.beta))


class SRM(nn.Module):
    # Lee et al. (2019): style pooling (mean, std) -> channel-wise FC -> BN -> sigmoid gate.
    def __init__(self, ch):
        super().__init__()
        self.cfc = nn.Parameter(torch.zeros(ch, 2))
        self.bn = nn.BatchNorm1d(ch)

    def forward(self, x):
        style = torch.stack([x.mean(dim=(2, 3)), (x.var(dim=(2, 3), unbiased=False) + 1e-5).sqrt()], dim=-1)
        return x * torch.sigmoid(self.bn((style * self.cfc).sum(-1)))[:, :, None, None]


class FCGammaBeta(nn.Module):
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(ch, reduction)
        self.fc_gamma = nn.Linear(reduction, ch)
        self.fc_beta = nn.Linear(reduction, ch)

    def forward(self, x):
        z = F.relu(self.fc1(x.mean(dim=(2, 3))))
        gamma = torch.sigmoid(self.fc_gamma(z))[:, :, None, None]
        beta = self.fc_beta(z)[:, :, None, None]
        return x * gamma + beta


class SNSMModulation(nn.Module):
    # Spectral-neighbourhood self-modulation (mean/var/gradient descriptors, hidden=8, normalised).
    # descs / zero_init exist only for the Phase 2 ablation; defaults are the paper's SNSM.
    def __init__(self, ch, hidden=8, use_beta=True, descs=('mean', 'var', 'grad'), zero_init=True):
        super().__init__()
        self.use_beta, self.descs = use_beta, descs
        self.spectral_conv1 = nn.Conv1d(len(descs), hidden, kernel_size=3, padding=2, dilation=2,
                                        padding_mode='replicate')
        self.spectral_conv2 = nn.Conv1d(hidden, 2 if use_beta else 1, kernel_size=3, padding=1,
                                        padding_mode='replicate')
        nn.init.kaiming_normal_(self.spectral_conv1.weight, mode='fan_out', nonlinearity='relu')
        if zero_init:
            nn.init.zeros_(self.spectral_conv2.weight)
            nn.init.zeros_(self.spectral_conv2.bias)

    @staticmethod
    def _norm(s):
        return (s - s.mean(dim=1, keepdim=True)) / (s.std(dim=1, keepdim=True) + 1e-6)

    def forward(self, x):
        mean = x.mean(dim=(2, 3))
        var = x.var(dim=(2, 3), unbiased=False)
        grad = torch.zeros_like(mean)
        grad[:, 1:-1] = (mean[:, 2:] - mean[:, :-2]) / 2.0
        grad[:, 0] = mean[:, 1] - mean[:, 0]
        grad[:, -1] = mean[:, -1] - mean[:, -2]
        d = {'mean': mean, 'var': var, 'grad': grad}
        stats = torch.stack([self._norm(d[k]) for k in self.descs], dim=1)
        z = self.spectral_conv2(F.relu(self.spectral_conv1(stats)))
        gamma = (1.0 + 0.5 * torch.tanh(z[:, 0]))[:, :, None, None]
        if not self.use_beta:
            return x * gamma
        return x * gamma + z[:, 1][:, :, None, None]


def make_modulation(kind, ch):
    return {'plain': lambda: None,
            'se_gamma': lambda: SEGamma(ch),
            'eca_gamma': lambda: ECAGamma(ch),
            'cbam': lambda: CBAM(ch),
            'gct': lambda: GCT(ch),
            'srm': lambda: SRM(ch),
            'fc_gamma_beta': lambda: FCGammaBeta(ch),
            'snsm_gamma': lambda: SNSMModulation(ch, use_beta=False),
            'snsm': lambda: SNSMModulation(ch, use_beta=True),
            'snsm_mean': lambda: SNSMModulation(ch, descs=('mean',)),
            'snsm_mean_var': lambda: SNSMModulation(ch, descs=('mean', 'var')),
            'snsm_mean_grad': lambda: SNSMModulation(ch, descs=('mean', 'grad')),
            'snsm_noinit': lambda: SNSMModulation(ch, zero_init=False)}[kind]()


class ResidualBlock(nn.Module):
    def __init__(self, ch, modulation):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(ch)
        self.mod = modulation
        nn.init.kaiming_normal_(self.conv1.weight, mode='fan_out', nonlinearity='relu')
        nn.init.kaiming_normal_(self.conv2.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        if self.mod is not None:
            out = self.mod(out)
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)


class ModulatedCNN(nn.Module):
    def __init__(self, num_classes, num_bands, kind):
        super().__init__()
        self.conv1 = nn.Conv2d(num_bands, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.block1 = ResidualBlock(64, make_modulation(kind, 64))
        self.block2 = ResidualBlock(64, make_modulation(kind, 64))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(64, num_classes)

    def features(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.block2(self.block1(x))
        return x.mean(dim=(2, 3))

    def classify(self, f):
        return self.fc(self.dropout(f))

    def forward(self, x):
        return self.classify(self.features(x))


class HybridSN(nn.Module):
    # Roy et al. (2020), original layer configuration (valid 3D convs -> 2D conv -> FC 256/128).
    def __init__(self, num_classes, num_bands):
        super().__init__()
        self.conv3d = nn.Sequential(
            nn.Conv3d(1, 8, (7, 3, 3)), nn.ReLU(),
            nn.Conv3d(8, 16, (5, 3, 3)), nn.ReLU(),
            nn.Conv3d(16, 32, (3, 3, 3)), nn.ReLU())
        d, s = num_bands - 12, WINDOW - 6
        self.conv2d = nn.Sequential(nn.Conv2d(32 * d, 64, 3), nn.ReLU())
        self.fc = nn.Sequential(
            nn.Linear(64 * (s - 2) ** 2, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.4))
        self.out = nn.Linear(128, num_classes)

    def features(self, x):
        x = self.conv3d(x.unsqueeze(1))
        x = self.conv2d(x.flatten(1, 2))
        return self.fc(x.flatten(1))

    def classify(self, f):
        return self.out(f)

    def forward(self, x):
        return self.classify(self.features(x))


class _Attention(nn.Module):
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.heads, self.scale = heads, (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        b, n, d = x.shape
        q, k, v = self.qkv(x).reshape(b, n, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        attn = ((q @ k.transpose(-2, -1)) * self.scale).softmax(dim=-1)
        return self.drop(self.proj((attn @ v).transpose(1, 2).reshape(b, n, d)))


class SSFTT(nn.Module):
    # Sun et al. (2022): 3D conv -> 2D conv -> Gaussian-weighted feature tokenizer (4 tokens)
    # -> 1 pre-norm transformer layer (8 heads, MLP dim 8) -> CLS token -> linear.
    def __init__(self, num_classes, num_bands, num_tokens=4, dim=64, heads=8, mlp_dim=8, dropout=0.1):
        super().__init__()
        self.conv3d = nn.Sequential(nn.Conv3d(1, 8, (3, 3, 3)), nn.BatchNorm3d(8), nn.ReLU())
        self.conv2d = nn.Sequential(nn.Conv2d(8 * (num_bands - 2), 64, 3), nn.BatchNorm2d(64), nn.ReLU())
        self.token_wA = nn.Parameter(torch.empty(1, num_tokens, 64))
        self.token_wV = nn.Parameter(torch.empty(1, 64, dim))
        nn.init.xavier_normal_(self.token_wA)
        nn.init.xavier_normal_(self.token_wV)
        self.pos_embedding = nn.Parameter(torch.empty(1, num_tokens + 1, dim))
        nn.init.normal_(self.pos_embedding, std=0.02)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.emb_drop = nn.Dropout(dropout)
        self.norm1, self.attn = nn.LayerNorm(dim), _Attention(dim, heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(mlp_dim, dim), nn.Dropout(dropout))
        self.head = nn.Linear(dim, num_classes)

    def features(self, x):
        x = self.conv3d(x.unsqueeze(1))
        x = self.conv2d(x.flatten(1, 2))
        x = x.flatten(2).transpose(1, 2)                                     # (B, HW, 64)
        A = torch.matmul(x, self.token_wA.transpose(1, 2)).transpose(1, 2)   # (B, L, HW)
        tokens = torch.matmul(A.softmax(dim=-1), torch.matmul(x, self.token_wV))
        x = torch.cat([self.cls_token.expand(x.size(0), -1, -1), tokens], dim=1)
        x = self.emb_drop(x + self.pos_embedding)
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x[:, 0]

    def classify(self, f):
        return self.head(f)

    def forward(self, x):
        return self.classify(self.features(x))


class DeepModulatedCNN(nn.Module):
    # Deeper, wider host network: 4 residual blocks of 128 channels (main backbone: 2 blocks of 64).
    def __init__(self, num_classes, num_bands, kind, width=128, depth=4):
        super().__init__()
        self.conv1 = nn.Conv2d(num_bands, width, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(width)
        self.blocks = nn.Sequential(*[ResidualBlock(width, make_modulation(kind, width)) for _ in range(depth)])
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(width, num_classes)

    def features(self, x):
        return self.blocks(F.relu(self.bn1(self.conv1(x)))).mean(dim=(2, 3))

    def classify(self, f):
        return self.fc(self.dropout(f))

    def forward(self, x):
        return self.classify(self.features(x))


def build_model(name, num_classes, num_bands):
    if name.startswith('deep_'):
        return DeepModulatedCNN(num_classes, num_bands, name[len('deep_'):])
    if name == 'hybridsn':
        return HybridSN(num_classes, num_bands)
    if name == 'ssftt':
        return SSFTT(num_classes, num_bands)
    return ModulatedCNN(num_classes, num_bands, name)

for _m in dict.fromkeys(m for s in STUDIES for m in s[3]):
    _net = build_model(_m, 16, 2 * PCA_COMPONENTS)
    _out = _net(torch.randn(2, 2 * PCA_COMPONENTS, WINDOW, WINDOW))
    print(f'{_m:14s} params={sum(p.numel() for p in _net.parameters()):>9,d} out={tuple(_out.shape)}')

In [ ]:
# =============================================================================
# Cell 4: Training, evaluation and efficiency
# =============================================================================
def train_and_evaluate(model, Xtr, ytr, Xte, yte, num_classes, seed):
    # Fixed-epoch training; the final model is evaluated once on the test set.
    Xtr_t = torch.from_numpy(Xtr).to(device)
    ytr_t = torch.from_numpy(ytr).to(device)
    gen = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=STEP_GAMMA)
    criterion = nn.CrossEntropyLoss()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(ytr_t), generator=gen).to(device)
        for i in range(0, len(perm), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            if len(idx) < 2:  # batch norm (e.g. SRM's BatchNorm1d) cannot train on a single sample
                continue
            optimizer.zero_grad()
            loss = criterion(model(Xtr_t[idx]), ytr_t[idx])
            loss.backward()
            optimizer.step()
        scheduler.step()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    train_time = time.time() - t0

    model.eval()
    preds, feats = [], []
    with torch.no_grad():
        for i in range(0, len(Xte), 1024):
            xb = torch.from_numpy(Xte[i:i + 1024]).to(device)
            f = model.features(xb)
            preds.append(model.classify(f).argmax(1).cpu().numpy())
            feats.append(f.cpu().numpy())
    del Xtr_t, ytr_t

    y_pred = np.concatenate(preds)
    cm = confusion_matrix(yte, y_pred, labels=list(range(num_classes)))
    support = cm.sum(axis=1)
    per_class = np.where(support > 0, np.diag(cm) / np.maximum(support, 1), np.nan) * 100
    return {
        'OA': float((y_pred == yte).mean() * 100),
        'AA': float(np.nanmean(per_class)),
        'kappa': float(cohen_kappa_score(yte, y_pred)),
        # Fisher is only comparable between the CNN variants (same 64-d GAP feature space).
        'fisher': fisher_ratio(np.concatenate(feats), yte),
        'train_time_s': train_time,
        'per_class': [None if np.isnan(a) else round(float(a), 4) for a in per_class],
    }


def fisher_ratio(features, labels):
    g = features.mean(axis=0)
    inter = intra = 0.0
    for c in np.unique(labels):
        fc = features[labels == c]
        mc = fc.mean(axis=0)
        inter += len(fc) * np.sum((mc - g) ** 2)
        intra += np.sum((fc - mc) ** 2)
    return float(inter / intra) if intra > 0 else 0.0


def measure_efficiency(model, num_bands):
    model.eval()
    params = sum(p.numel() for p in model.parameters())
    dummy = torch.randn(1, num_bands, WINDOW, WINDOW, device=device)
    macs = None  # thop counts MACs of conv/linear layers only; matmul/mean/var ops are not included
    if thop is not None:
        try:
            macs, _ = thop.profile(copy.deepcopy(model), inputs=(dummy,), verbose=False)
        except Exception as e:
            print(f'    thop failed: {e}')
    times = []
    with torch.no_grad():
        for _ in range(10):
            model(dummy)
        for _ in range(100):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.time()
            model(dummy)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            times.append(time.time() - t0)
    return {'params': int(params), 'macs_convlinear': None if macs is None else float(macs),
            'latency_ms': float(np.mean(times) * 1000)}

print('Training / evaluation utilities ready.')

In [ ]:
# =============================================================================
# Cell 5: Run the Phase 8 studies (resumable within a session)
# =============================================================================
RUNS_CSV = os.path.join(OUT_DIR, 'runs.csv')
done = set()
if os.path.exists(RUNS_CSV):
    _prev = pd.read_csv(RUNS_CSV)
    done = set(zip(_prev.study, _prev.model, _prev.seed))

GATES_CSV = os.path.join(OUT_DIR, 'gct_gates.csv')


def log_gct_gates(model, X, name, m, seed, n=2048):
    # Records how far each trained GCT gate 1 + tanh(.) moves from the identity (gate = 1) on test patches.
    rows, hooks = [], []

    def hook(mod, inp, out, layer):
        x = inp[0]
        emb = (x.pow(2).sum(dim=(2, 3), keepdim=True) + mod.eps).sqrt() * mod.alpha
        norm = mod.gamma / (emb.pow(2).mean(dim=1, keepdim=True) + mod.eps).sqrt()
        g = (1.0 + torch.tanh(emb * norm + mod.beta)).flatten(1)
        rows.append({'study': name, 'model': m, 'seed': seed, 'layer': layer,
                     'gate_mean': g.mean().item(), 'gate_sd_channels': g.std(dim=1).mean().item(),
                     'gate_abs_dev': (g - 1).abs().mean().item(), 'gamma_abs': mod.gamma.abs().mean().item()})

    gcts = [mod for mod in model.modules() if isinstance(mod, GCT)]
    for k, mod in enumerate(gcts):
        hooks.append(mod.register_forward_hook(lambda mod, i, o, k=k: hook(mod, i, o, k)))
    model.eval()
    with torch.no_grad():
        model(torch.from_numpy(X[:n]).to(device))
    for h in hooks:
        h.remove()
    pd.DataFrame(rows).to_csv(GATES_CSV, mode='a', header=not os.path.exists(GATES_CSV), index=False)


cache = {}
t_start = time.time()
for name, ds, split, models, seeds, lr, feats_kind in STUDIES:
    if all((name, m, s) in done for s in seeds for m in models):
        continue
    if ds not in cache:
        img, gt = load_dataset(ds)
        filtered = wiener_denoise(img)
        del img
        scene = wavelet_features(apply_pca(filtered, PCA_COMPONENTS))   # = preprocess(img)
        cache = {ds: (gt, filtered, pad(scene))}                         # keep one scene in memory
    gt, filtered, padded_scene = cache[ds]
    num_classes = int(gt.max())
    print(f'\n=== {name}: {ds}, {split}, lr={lr}, features={feats_kind} ===')
    for seed in seeds:
        pending = [m for m in models if (name, m, seed) not in done]
        if not pending:
            continue
        if split == 'random':
            tr_xy, ytr, te_xy, yte = split_centers(gt, seed)
        elif split == 'blocks':
            tr_xy, ytr, te_xy, yte = cap_test(split_blocks(gt, seed), seed)
        elif split == 'disjoint':
            tr_xy, ytr, te_xy, yte = cap_test(split_disjoint(gt, seed), seed)
        else:
            tr_xy, ytr, te_xy, yte = split_hard(gt, seed, HARD[ds])
        padded = padded_scene if feats_kind == 'scene' else pad(features_train_pca(filtered, tr_xy))
        Xtr, Xte = extract_patches(padded, tr_xy), extract_patches(padded, te_xy)
        LR = lr   # train_and_evaluate reads the module-level LR
        for m in pending:
            set_seed(seed)
            model = build_model(m, num_classes, padded.shape[2]).to(device)
            r = train_and_evaluate(model, Xtr, ytr, Xte, yte, num_classes, seed)
            pd.DataFrame([{'study': name, 'dataset': ds, 'model': m, 'seed': seed, 'lr': lr, 'features': feats_kind,
                           'n_train': len(ytr), 'n_test': len(yte), 'n_test_classes': int(len(np.unique(yte))),
                           'OA': r['OA'], 'AA': r['AA'], 'kappa': r['kappa'], 'train_time_s': r['train_time_s'],
                           'per_class': json.dumps(r['per_class'])}]).to_csv(
                RUNS_CSV, mode='a', header=not os.path.exists(RUNS_CSV), index=False)
            done.add((name, m, seed))
            if m.endswith('gct'):
                log_gct_gates(model, Xte, name, m, seed)
            print(f'  seed {seed} {m:16s} OA={r["OA"]:6.2f} AA={r["AA"]:6.2f} '
                  f'(elapsed {(time.time() - t_start) / 60:.1f} min)')
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        del Xtr, Xte
        gc.collect()
LR = 1e-3
print(f'\nAll Phase 8 runs finished in {(time.time() - t_start) / 60:.1f} min')


In [ ]:
# =============================================================================
# Cell 6: Summary (mean ± sample std over seeds)
# =============================================================================
runs = pd.read_csv(RUNS_CSV)
summary = (runs.groupby(['study', 'dataset', 'model'])
               .agg(n=('seed', 'size'), n_train=('n_train', 'mean'), n_test=('n_test', 'mean'),
                    OA=('OA', 'mean'), OA_std=('OA', 'std'), AA=('AA', 'mean'), AA_std=('AA', 'std'))
               .round(2).reset_index())
summary.to_csv(os.path.join(OUT_DIR, 'summary.csv'), index=False)
print(summary.to_string(index=False))
